# Results exploration

Interrogate a completed run before drafting Chapter 4. Do not paste output from the synthetic county into the dissertation.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from kajiado_lceo import load_config
from kajiado_lceo.pipeline import run

config = load_config("../config", scenario="baseline")
config.scenario

In [ ]:
from kajiado_lceo.sample import generate_sample_county

result = run(config, inputs=generate_sample_county(seed=42), write_outputs=False)
settlements = result.settlements
result.summary["totals"]

## Technology mix by typology

Table 4.2 in outline.

In [ ]:
pd.crosstab(settlements["typology"], settlements["least_cost_technology"],
            normalize="index").round(3)

## How was each settlement decided?

A plan resting heavily on `dispersed_override` rests on assumption rather than on comparison. The reader is entitled to know how heavily.

In [ ]:
settlements["decision_reason"].value_counts()

## Robustness of each choice

`lcoe_margin` is the gap to the runner-up. A small margin means the choice would flip under modest parameter error.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
settlements["lcoe_margin"].dropna().clip(upper=1.0).hist(bins=50, ax=ax)
ax.set_xlabel("LCOE gap to runner-up (USD/kWh)")
ax.set_ylabel("Settlements"); plt.show()

## Viability and the subsidy gap

In [ ]:
for key in config.section("finance.perspectives"):
    viable = settlements[f"viable_{key}"].sum()
    gap = settlements[f"subsidy_gap_{key}"].sum()
    print(f"{key:10s}: {viable:4d} viable, subsidy gap USD {gap:,.0f}")

## Marginal cost of access

Households reached against capital committed, following the priority ranking. Figure 4.5 in outline.

In [ ]:
plan = result.priority_plan
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plan["cumulative_capex_usd"] / 1e6, plan["cumulative_households"])
ax.set_xlabel("Cumulative CAPEX (USD millions)")
ax.set_ylabel("Cumulative households reached")
ax.grid(alpha=0.3); plt.show()